In [1]:
import os
import argparse
import numpy as np
import torch
import torch.nn as nn
from PIL import Image
from datasets import Dataset, DatasetDict, load_from_disk
from concurrent.futures import ThreadPoolExecutor
from torchvision.transforms import ColorJitter, RandomHorizontalFlip, RandomVerticalFlip
from transformers import (
    SegformerForSemanticSegmentation,
    SegformerImageProcessor,
    UperNetForSemanticSegmentation,
    AutoImageProcessor,
    TrainingArguments,
    Trainer,
)
import evaluate
import json

In [2]:
# GPU 환경 확인
print("=" * 60)
print("[환경 확인]")
print(f"  CUDA 사용 가능: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"  GPU 이름: {torch.cuda.get_device_name()}")
    vram_total = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)
    print(f"  VRAM 총량: {vram_total:.1f} GB")
    torch.cuda.empty_cache()
print("=" * 60)

[환경 확인]
  CUDA 사용 가능: True
  GPU 이름: NVIDIA GeForce RTX 4060
  VRAM 총량: 8.0 GB


In [3]:
# 설정값
MODEL_TYPE = "upernet"  # "segformer_b0" / "segformer_b2" / "upernet"
IMAGE_DIR = r"D:\Work\study_full_data\image_segmentation\Semantic_Segmentation_Drone_Dataset\semantic_drone_dataset\semantic_drone_dataset\original_images"
LABEL_DIR = r"D:\Work\study_full_data\image_segmentation\Semantic_Segmentation_Drone_Dataset\semantic_drone_dataset\semantic_drone_dataset\label_images_semantic"
OUTPUT_DIR = r"D:\Work\study_full_data\image_segmentation\output"
EPOCHS = 50
BATCH_SIZE = 4
LEARNING_RATE = 3e-5

In [4]:
# 클래스 정의 (24 클래스)
id2label = {
    0: "unlabeled",
    1: "paved-area",
    2: "dirt",
    3: "grass",
    4: "gravel",
    5: "water",
    6: "rocks",
    7: "pool",
    8: "vegetation",
    9: "roof",
    10: "wall",
    11: "window",
    12: "door",
    13: "fence",
    14: "fence-pole",
    15: "person",
    16: "dog",
    17: "car",
    18: "bicycle",
    19: "tree",
    20: "bald-tree",
    21: "ar-marker",
    22: "obstacle",
    23: "conflicting",
}
label2id = {v: k for k, v in id2label.items()}
num_labels = len(id2label)

In [5]:
# RGB 컬러맵 (시각화용)
LABEL_COLORMAP = {
    0: (0, 0, 0),
    1: (128, 64, 128),
    2: (130, 76, 0),
    3: (0, 102, 0),
    4: (112, 103, 87),
    5: (28, 42, 168),
    6: (48, 41, 30),
    7: (0, 50, 89),
    8: (107, 142, 35),
    9: (70, 70, 70),
    10: (102, 102, 156),
    11: (254, 228, 12),
    12: (254, 148, 12),
    13: (190, 153, 153),
    14: (153, 153, 153),
    15: (255, 22, 96),
    16: (102, 51, 0),
    17: (9, 143, 150),
    18: (119, 11, 32),
    19: (51, 51, 0),
    20: (190, 250, 190),
    21: (112, 150, 146),
    22: (2, 135, 115),
    23: (255, 0, 0),
}

In [6]:
# 데이터 로딩 (리사이즈 포함)
TARGET_SIZE = (512, 512)
 
 
def process_image(image_file, image_directory, label_directory):
    file_name, _ = os.path.splitext(image_file)
    label_file = f"{file_name}.png"
 
    label_path = os.path.join(label_directory, label_file)
    if not os.path.exists(label_path):
        return None
 
    with Image.open(os.path.join(image_directory, image_file)) as im:
        img = im.convert("RGB").resize(TARGET_SIZE, Image.BILINEAR)
 
    with Image.open(label_path) as im:
        # R=G=B 형태이므로 L 변환 후 NEAREST 리사이즈 (클래스 ID 보존)
        lbl = im.convert("L").resize(TARGET_SIZE, Image.NEAREST)
 
    return {"pixel_values": img, "label": lbl, "filename": image_file}
 
 
def load_images_and_labels(image_directory, label_directory):
    image_files = sorted([
        f for f in os.listdir(image_directory)
        if f.lower().endswith((".jpg", ".jpeg", ".png"))
    ])
 
    data = []
    with ThreadPoolExecutor(max_workers=4) as executor:
        results = executor.map(
            process_image,
            image_files,
            [image_directory] * len(image_files),
            [label_directory] * len(image_files),
        )
        for result in results:
            if result is not None:
                data.append(result)
 
    print(f"Loaded {len(data)} image-label pairs")
    return data
 
 
def create_dataset(image_directory, label_directory):
    data = load_images_and_labels(image_directory, label_directory)
    dataset = Dataset.from_dict({
        "pixel_values": [item["pixel_values"] for item in data],
        "label": [item["label"] for item in data],
        "filename": [item["filename"] for item in data],
    })
    return dataset

In [7]:
# 전처리 및 데이터 증강
def get_transforms(model_type="segformer"):
    """모델 타입에 따라 적절한 이미지 프로세서와 변환 반환"""
 
    if model_type.startswith("segformer"):
        processor = SegformerImageProcessor(
            do_resize=True,
            size={"height": 512, "width": 512},
            do_reduce_labels=False,
        )
    elif model_type == "upernet":
        processor = AutoImageProcessor.from_pretrained(
            "openmmlab/upernet-swin-tiny",
            do_resize=True,
            size={"height": 512, "width": 512},
            do_reduce_labels=False,
        )
    else:
        raise ValueError(f"Unknown model_type: {model_type}")
 
    jitter = ColorJitter(brightness=0.25, contrast=0.25, saturation=0.25, hue=0.1)
    h_flip = RandomHorizontalFlip(p=0.5)
    v_flip = RandomVerticalFlip(p=0.5)
 
    def train_transforms(example_batch):
        images = []
        labels = []
        for img, lbl in zip(example_batch["pixel_values"], example_batch["label"]):
            # 동일한 시드로 이미지/레이블에 같은 flip 적용
            seed = torch.randint(0, 2**32, (1,)).item()
 
            img = jitter(img)
 
            torch.manual_seed(seed)
            img = h_flip(img)
            torch.manual_seed(seed)
            lbl = h_flip(lbl)
 
            torch.manual_seed(seed + 1)
            img = v_flip(img)
            torch.manual_seed(seed + 1)
            lbl = v_flip(lbl)
 
            images.append(img)
            labels.append(lbl)
 
        inputs = processor(images, labels)
        return inputs
 
    def val_transforms(example_batch):
        images = [x for x in example_batch["pixel_values"]]
        labels = [x for x in example_batch["label"]]
        inputs = processor(images, labels)
        return inputs
 
    return processor, train_transforms, val_transforms

In [8]:
# 모델 생성
def create_model(model_type="segformer_b0"):
    if model_type == "segformer_b0":
        model = SegformerForSemanticSegmentation.from_pretrained(
            "nvidia/mit-b0",
            id2label=id2label,
            label2id=label2id,
        )
    elif model_type == "segformer_b2":
        model = SegformerForSemanticSegmentation.from_pretrained(
            "nvidia/mit-b2",
            id2label=id2label,
            label2id=label2id,
        )
    elif model_type == "upernet":
        model = UperNetForSemanticSegmentation.from_pretrained(
            "openmmlab/upernet-swin-tiny",
            id2label=id2label,
            label2id=label2id,
            ignore_mismatched_sizes=True,
        )
    else:
        raise ValueError(f"Unknown model_type: {model_type}")
 
    return model

In [9]:
# 평가 메트릭
def get_compute_metrics():
    iou_metric = evaluate.load("mean_iou")
 
    def compute_metrics(eval_pred):
        with torch.no_grad():
            logits, labels = eval_pred
            logits_tensor = torch.from_numpy(logits)
 
            resized_logits = nn.functional.interpolate(
                logits_tensor,
                size=labels.shape[-2:],
                mode="bilinear",
                align_corners=False,
            ).argmax(dim=1)
 
            pred_labels = resized_logits.detach().cpu().numpy()
 
            metrics = iou_metric._compute(
                predictions=pred_labels,
                references=labels,
                num_labels=num_labels,
                ignore_index=0,  # unlabeled 무시
                reduce_labels=False,
            )
 
            per_cat_accuracy = metrics.pop("per_category_accuracy").tolist()
            per_cat_iou = metrics.pop("per_category_iou").tolist()
 
            metrics.update({
                f"accuracy_{id2label[i]}": v
                for i, v in enumerate(per_cat_accuracy)
            })
            metrics.update({
                f"iou_{id2label[i]}": v
                for i, v in enumerate(per_cat_iou)
            })
 
        return metrics
 
    return compute_metrics

In [10]:
# 메인 학습 함수
def train():
    print(f"\n{'='*60}")
    print(f"  Model: {MODEL_TYPE}")
    print(f"  Epochs: {EPOCHS}")
    print(f"  Batch size: {BATCH_SIZE}")
    print(f"  Learning rate: {LEARNING_RATE}")
    print(f"  Image size: {TARGET_SIZE}")
    print(f"{'='*60}\n")
 
    os.makedirs(OUTPUT_DIR, exist_ok=True)
 
    # 데이터셋 로드/생성
    dataset_cache = os.path.join(OUTPUT_DIR, "dataset_cache.hf")
    if os.path.exists(dataset_cache):
        print("Loading cached dataset...")
        full_dataset = load_from_disk(dataset_cache)
    else:
        print("Creating dataset...")
        full_dataset = create_dataset(IMAGE_DIR, LABEL_DIR)
        full_dataset.save_to_disk(dataset_cache)
        print(f"Dataset cached to {dataset_cache}")
 
    # Train/Val/Test 분할 (80:10:10)
    split = full_dataset.train_test_split(test_size=0.2, seed=42)
    train_ds = split["train"]
    val_test = split["test"].train_test_split(test_size=0.5, seed=42)
    val_ds = val_test["train"]
    test_ds = val_test["test"]
    print(f"Train: {len(train_ds)}, Val: {len(val_ds)}, Test: {len(test_ds)}")
 
    # 분할 파일명 목록 저장 (평가 시 사용)
    split_info = {
        "train_files": train_ds["filename"],
        "val_files": val_ds["filename"],
        "test_files": test_ds["filename"],
    }
    split_path = os.path.join(OUTPUT_DIR, "split_info.json")
    with open(split_path, "w") as f:
        json.dump(split_info, f, indent=2)
    print(f"Split info saved to {split_path}")
 
    # 전처리 설정
    model_category = "upernet" if MODEL_TYPE == "upernet" else "segformer"
    processor, train_tf, val_tf = get_transforms(model_category)
    train_ds.set_transform(train_tf)
    val_ds.set_transform(val_tf)
    test_ds.set_transform(val_tf)
 
    # 모델 생성
    model = create_model(MODEL_TYPE)
    print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")
 
    # 학습 설정
    run_dir = os.path.join(OUTPUT_DIR, f"run_{MODEL_TYPE}")
    training_args = TrainingArguments(
        output_dir=run_dir,
        learning_rate=LEARNING_RATE,
        num_train_epochs=EPOCHS,
        per_device_train_batch_size=BATCH_SIZE,
        per_device_eval_batch_size=BATCH_SIZE,
        save_total_limit=3,
        # evaluation_strategy="epoch",
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=10,
        eval_accumulation_steps=10,
        load_best_model_at_end=True,
        metric_for_best_model="mean_iou",
        greater_is_better=True,
        fp16=True,  # RTX 4060 mixed precision
        # dataloader_num_workers=2,
        dataloader_num_workers=0,
        remove_unused_columns=False,
    )
 
    # Trainer 설정 및 학습
    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_ds,
        eval_dataset=test_ds,
        compute_metrics=get_compute_metrics(),
    )
 
    trainer.train()

    # 노트북 진행 표시줄 콜백 제거 (오류 방지)
    from transformers.utils.notebook import NotebookProgressCallback
    trainer.remove_callback(NotebookProgressCallback)
 
    # 최종 평가 (테스트셋 - 학습 중 미사용 데이터)
    print("\n" + "=" * 60)
    print("  Final Evaluation on Test Set (unseen during training)")
    print("=" * 60)
    eval_results = trainer.evaluate(eval_dataset=test_ds)
 
    # 주요 메트릭 출력
    print(f"\n  Mean IoU:      {eval_results.get('eval_mean_iou', 'N/A'):.4f}")
    print(f"  Mean Accuracy: {eval_results.get('eval_mean_accuracy', 'N/A'):.4f}")
    print(f"  Overall Acc:   {eval_results.get('eval_overall_accuracy', 'N/A'):.4f}")
 
    # 카테고리별 IoU 출력
    print(f"\n  {'Category':<15} {'IoU':>8} {'Accuracy':>10}")
    print(f"  {'-'*35}")
    for i in range(num_labels):
        cat = id2label[i]
        iou = eval_results.get(f"eval_iou_{cat}", float("nan"))
        acc = eval_results.get(f"eval_accuracy_{cat}", float("nan"))
        print(f"  {cat:<15} {iou:>8.4f} {acc:>10.4f}")
 
    # 모델 저장
    save_dir = os.path.join(OUTPUT_DIR, f"model_{MODEL_TYPE}")
    model.save_pretrained(save_dir)
    processor_to_save = (
        SegformerImageProcessor.from_pretrained("nvidia/mit-b0")
        if model_category == "segformer"
        else AutoImageProcessor.from_pretrained("openmmlab/upernet-swin-tiny")
    )
    processor_to_save.save_pretrained(save_dir)
    print(f"\nModel saved to {save_dir}")
 
    # 학습 로그 저장
    log_path = os.path.join(OUTPUT_DIR, f"log_{MODEL_TYPE}.json")
    with open(log_path, "w") as f:
        json.dump(trainer.state.log_history, f, indent=2)
    print(f"Training log saved to {log_path}")
 
    # 평가 결과 저장
    eval_path = os.path.join(OUTPUT_DIR, f"eval_{MODEL_TYPE}.json")
    with open(eval_path, "w") as f:
        json.dump(eval_results, f, indent=2)
    print(f"Eval results saved to {eval_path}")
 
    return eval_results

In [11]:
train()


  Model: upernet
  Epochs: 50
  Batch size: 4
  Learning rate: 3e-05
  Image size: (512, 512)

Loading cached dataset...
Train: 320, Val: 40, Test: 40
Split info saved to D:\Work\study_full_data\image_segmentation\output\split_info.json


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
Some weights of UperNetForSemanticSegmentation were not initialized from the model checkpoint at openmmlab/upernet-swin-tiny and are newly initialized because the shapes did not match:
- decode_head.classifier.weight: found shape torch.Size([150, 512, 1, 1]) in the checkpoint and torch.Size([24, 512, 1, 1]) in the model instantiated
- decode_head.classifier.bias: found shape torch.Size([150]) in the checkpoint and torch.Size([24]) in the model instantiated
- auxiliary_head.classifier.weight: found shape torch.Size([150, 256, 1, 1]) in the checkpoint and torch.Size([24, 256, 1, 1]) in the model instantiated
- auxiliary_head.classifier.bias: found shape torch.Size([150]

Model parameters: 59,844,842


Epoch,Training Loss,Validation Loss,Mean Iou,Mean Accuracy,Overall Accuracy,Accuracy Unlabeled,Accuracy Paved-area,Accuracy Dirt,Accuracy Grass,Accuracy Gravel,Accuracy Water,Accuracy Rocks,Accuracy Pool,Accuracy Vegetation,Accuracy Roof,Accuracy Wall,Accuracy Window,Accuracy Door,Accuracy Fence,Accuracy Fence-pole,Accuracy Person,Accuracy Dog,Accuracy Car,Accuracy Bicycle,Accuracy Tree,Accuracy Bald-tree,Accuracy Ar-marker,Accuracy Obstacle,Accuracy Conflicting,Iou Unlabeled,Iou Paved-area,Iou Dirt,Iou Grass,Iou Gravel,Iou Water,Iou Rocks,Iou Pool,Iou Vegetation,Iou Roof,Iou Wall,Iou Window,Iou Door,Iou Fence,Iou Fence-pole,Iou Person,Iou Dog,Iou Car,Iou Bicycle,Iou Tree,Iou Bald-tree,Iou Ar-marker,Iou Obstacle,Iou Conflicting
1,1.948100,1.739822,0.213779,0.283791,0.722983,nan,0.777900,0.166804,0.991193,0.738190,0.769261,0.000000,0.000000,0.781886,0.936302,0.140367,0.000000,0.000000,0.000000,0.000000,0.013692,nan,0.287084,0.000000,0.100472,0.007089,0.000000,0.249370,nan,nan,0.693954,0.159711,0.766323,0.445776,0.671718,0.000000,0.000000,0.504818,0.560904,0.118312,0.000000,0.000000,0.000000,0.000000,0.013628,nan,0.286507,0.000000,0.091024,0.006930,0.000000,0.169743,nan
2,1.205200,1.167271,0.354544,0.432512,0.791940,nan,0.867514,0.451443,0.985286,0.777821,0.939266,0.000000,0.675344,0.673079,0.940152,0.491425,0.000000,0.000000,0.000880,0.000000,0.371480,nan,0.739188,0.000000,0.570578,0.320466,0.000000,0.278822,nan,nan,0.786180,0.356702,0.857727,0.460306,0.806043,0.000000,0.665809,0.521111,0.809622,0.302520,0.000000,0.000000,0.000877,0.000000,0.335080,nan,0.712978,0.000000,0.335209,0.271747,0.000000,0.223521,nan
3,1.037100,0.875602,0.450534,0.533535,0.852328,nan,0.939758,0.596709,0.966853,0.779342,0.977144,0.097501,0.955735,0.806023,0.947454,0.620396,0.081130,0.000000,0.068672,0.000000,0.638704,nan,0.856993,0.168915,0.598335,0.663490,0.000000,0.441077,nan,nan,0.882845,0.423590,0.906205,0.616687,0.839269,0.093367,0.926851,0.617763,0.885793,0.386844,0.078459,0.000000,0.066869,0.000000,0.500969,nan,0.798595,0.157390,0.471339,0.471871,0.000000,0.336511,nan
4,0.834400,0.782946,0.464740,0.548194,0.855078,nan,0.953958,0.604045,0.975803,0.795101,0.975285,0.258987,0.931887,0.706130,0.936625,0.692198,0.159857,0.000000,0.142346,0.000000,0.641135,nan,0.899806,0.213786,0.703997,0.420484,0.000000,0.500650,nan,nan,0.890636,0.457037,0.906939,0.620216,0.851646,0.204133,0.914665,0.577637,0.911984,0.426829,0.144022,0.000000,0.124147,0.000000,0.520083,nan,0.831876,0.195844,0.437814,0.353997,0.000000,0.390040,nan
5,0.744800,0.694818,0.510748,0.590728,0.872016,nan,0.958786,0.572124,0.973614,0.791101,0.962307,0.321489,0.973600,0.834621,0.963630,0.672297,0.205194,0.000000,0.241885,0.000000,0.735525,nan,0.906021,0.294610,0.704414,0.654632,0.227475,0.411961,nan,nan,0.895449,0.450677,0.913090,0.650876,0.870590,0.253651,0.939345,0.672372,0.891168,0.455534,0.170844,0.000000,0.205080,0.000000,0.572765,nan,0.851818,0.254253,0.597160,0.505520,0.227003,0.348512,nan
6,0.798700,0.660802,0.531494,0.626403,0.871615,nan,0.965773,0.597850,0.974375,0.827283,0.969605,0.429865,0.982447,0.701998,0.945105,0.656718,0.464522,0.000000,0.329406,0.000000,0.706660,nan,0.916276,0.317018,0.809355,0.644868,0.386176,0.529172,nan,nan,0.894692,0.502709,0.918529,0.667021,0.878585,0.292185,0.948894,0.620436,0.923760,0.476805,0.290301,0.000000,0.256403,0.000000,0.577281,nan,0.876343,0.279013,0.463554,0.490213,0.380264,0.424389,nan
7,0.686000,0.605587,0.552586,0.635791,0.883039,nan,0.962164,0.596149,0.972540,0.812742,0.970152,0.447107,0.961690,0.869416,0.950695,0.678266,0.406095,0.000000,0.422748,0.000000,0.670152,nan,0.916819,0.336288,0.691099,0.669474,0.531989,0.486016,nan,nan,0.901446,0.491652,0.923423,0.672387,0.883747,0.343439,0.941167,0.709941,0.911740,0.480266,0.272893,0.000000,0.294965,0.000000,0.566858,nan,0.873035,0.295229,0.616109,0.505343,0.513873,0.406786,nan
8,0.627600,0.588744,0.562193,0.654161,0.885195,nan,0.956514,0.599843,0.969456,0.811636,0.959842,0.519338,0.983581,0.881

C:\Users\fbwlg\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-metric--mean_iou\9e450724f21f05592bfb0255fe2fa576df8171fa060d11121d8aecfff0db80d0\mean_iou.py:259: RuntimeWarning: invalid value encountered in divide
  iou = total_area_intersect / total_area_union
C:\Users\fbwlg\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-metric--mean_iou\9e450724f21f05592bfb0255fe2fa576df8171fa060d11121d8aecfff0db80d0\mean_iou.py:260: RuntimeWarning: invalid value encountered in divide
  acc = total_area_intersect / total_area_label
C:\Users\fbwlg\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-metric--mean_iou\9e450724f21f05592bfb0255fe2fa576df8171fa060d11121d8aecfff0db80d0\mean_iou.py:259: RuntimeWarning: invalid value encountered in divide
  iou = total_area_intersect / total_area_union
C:\Users\fbwlg\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-metric--mean_iou\9e450724f21f05592bfb0255fe2fa576df8171fa060d11121d8aecfff0db80d0\mean_iou.py:260: 


  Final Evaluation on Test Set (unseen during training)


C:\Users\fbwlg\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-metric--mean_iou\9e450724f21f05592bfb0255fe2fa576df8171fa060d11121d8aecfff0db80d0\mean_iou.py:259: RuntimeWarning: invalid value encountered in divide
  iou = total_area_intersect / total_area_union
C:\Users\fbwlg\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-metric--mean_iou\9e450724f21f05592bfb0255fe2fa576df8171fa060d11121d8aecfff0db80d0\mean_iou.py:260: RuntimeWarning: invalid value encountered in divide
  acc = total_area_intersect / total_area_label



  Mean IoU:      0.6091
  Mean Accuracy: 0.7361
  Overall Acc:   0.9123

  Category             IoU   Accuracy
  -----------------------------------
  unlabeled         0.0000        nan
  paved-area        0.9306     0.9672
  dirt              0.5892     0.7167
  grass             0.9398     0.9707
  gravel            0.7223     0.8428
  water             0.9178     0.9586
  rocks             0.4483     0.6835
  pool              0.9631     0.9960
  vegetation        0.7698     0.8846
  roof              0.9531     0.9696
  wall              0.6162     0.7530
  window            0.4129     0.6335
  door              0.0000     0.0000
  fence             0.4989     0.6199
  fence-pole        0.0000     0.0000
  person            0.6704     0.8189
  dog                  nan        nan
  car               0.9126     0.9538
  bicycle           0.4374     0.5899
  tree              0.6858     0.7976
  bald-tree         0.5519     0.7469
  ar-marker         0.8108     0.8598
  obstacle    

{'eval_loss': 0.42773300409317017,
 'eval_mean_iou': 0.6091154803332662,
 'eval_mean_accuracy': 0.736121508642167,
 'eval_overall_accuracy': 0.9122876872324049,
 'eval_accuracy_unlabeled': nan,
 'eval_accuracy_paved-area': 0.9672059208389046,
 'eval_accuracy_dirt': 0.7166747580591502,
 'eval_accuracy_grass': 0.9707492201273815,
 'eval_accuracy_gravel': 0.842828764402066,
 'eval_accuracy_water': 0.9586257403044004,
 'eval_accuracy_rocks': 0.6835330406928461,
 'eval_accuracy_pool': 0.9960017014036581,
 'eval_accuracy_vegetation': 0.8845596210843392,
 'eval_accuracy_roof': 0.9696460035239468,
 'eval_accuracy_wall': 0.7529991225677061,
 'eval_accuracy_window': 0.6335487784959283,
 'eval_accuracy_door': 0.0,
 'eval_accuracy_fence': 0.6198999762789594,
 'eval_accuracy_fence-pole': 0.0,
 'eval_accuracy_person': 0.8188700898158258,
 'eval_accuracy_dog': nan,
 'eval_accuracy_car': 0.9537803706991829,
 'eval_accuracy_bicycle': 0.5898805263447668,
 'eval_accuracy_tree': 0.7975517850314298,
 'eval

In [12]:
MODEL_TYPE = "segformer_b0"
IMAGE_DIR = r"D:\Work\study_full_data\image_segmentation\Semantic_Segmentation_Drone_Dataset\semantic_drone_dataset\semantic_drone_dataset\original_images"
LABEL_DIR = r"D:\Work\study_full_data\image_segmentation\Semantic_Segmentation_Drone_Dataset\semantic_drone_dataset\semantic_drone_dataset\label_images_semantic"
OUTPUT_DIR = r"D:\Work\study_full_data\image_segmentation\output\segformer_b0"
EPOCHS = 50
BATCH_SIZE = 4
LEARNING_RATE = 3e-5

In [13]:
train()


  Model: segformer_b0
  Epochs: 50
  Batch size: 4
  Learning rate: 3e-05
  Image size: (512, 512)

Creating dataset...
Loaded 400 image-label pairs


Saving the dataset (0/1 shards):   0%|          | 0/400 [00:00<?, ? examples/s]

Dataset cached to D:\Work\study_full_data\image_segmentation\output\segformer_b0\dataset_cache.hf
Train: 320, Val: 40, Test: 40
Split info saved to D:\Work\study_full_data\image_segmentation\output\segformer_b0\split_info.json


Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/mit-b0 and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch_norm.weight', 'decode_head.classifier.bias', 'decode_head.classifier.weight', 'decode_head.linear_c.0.proj.bias', 'decode_head.linear_c.0.proj.weight', 'decode_head.linear_c.1.proj.bias', 'decode_head.linear_c.1.proj.weight', 'decode_head.linear_c.2.proj.bias', 'decode_head.linear_c.2.proj.weight', 'decode_head.linear_c.3.proj.bias', 'decode_head.linear_c.3.proj.weight', 'decode_head.linear_fuse.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model parameters: 3,720,312


...


  Final Evaluation on Test Set (unseen during training)


C:\Users\fbwlg\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-metric--mean_iou\9e450724f21f05592bfb0255fe2fa576df8171fa060d11121d8aecfff0db80d0\mean_iou.py:259: RuntimeWarning: invalid value encountered in divide
  iou = total_area_intersect / total_area_union
C:\Users\fbwlg\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-metric--mean_iou\9e450724f21f05592bfb0255fe2fa576df8171fa060d11121d8aecfff0db80d0\mean_iou.py:260: RuntimeWarning: invalid value encountered in divide
  acc = total_area_intersect / total_area_label



  Mean IoU:      0.5015
  Mean Accuracy: 0.5843
  Overall Acc:   0.8756

  Category             IoU   Accuracy
  -----------------------------------
  unlabeled            nan        nan
  paved-area        0.8913     0.9598
  dirt              0.4978     0.6278
  grass             0.9164     0.9654
  gravel            0.6660     0.8150
  water             0.8694     0.9509
  rocks             0.1745     0.1992
  pool              0.8858     0.9833
  vegetation        0.7127     0.8538
  roof              0.8980     0.9327
  wall              0.4437     0.6457
  window            0.0394     0.0405
  door              0.0000     0.0000
  fence             0.2434     0.2971
  fence-pole        0.0000     0.0000
  person            0.5281     0.6784
  dog                  nan        nan
  car               0.8548     0.9222
  bicycle           0.2330     0.2533
  tree              0.5843     0.7187
  bald-tree         0.5088     0.7322
  ar-marker         0.1860     0.1870
  obstacle    

D:\ProgramData\anaconda3\envs\transformer_learn\Lib\site-packages\transformers\utils\deprecation.py:172: UserWarning: The following named arguments are not valid for `SegformerImageProcessor.__init__` and were ignored: 'feature_extractor_type'
  return func(*args, **kwargs)


{'eval_loss': 0.42604392766952515,
 'eval_mean_iou': 0.5015435089067312,
 'eval_mean_accuracy': 0.5843092095454816,
 'eval_overall_accuracy': 0.8756227947833227,
 'eval_accuracy_unlabeled': nan,
 'eval_accuracy_paved-area': 0.9597634059048565,
 'eval_accuracy_dirt': 0.627841126124683,
 'eval_accuracy_grass': 0.9653654583620004,
 'eval_accuracy_gravel': 0.8150464839094159,
 'eval_accuracy_water': 0.950932942749793,
 'eval_accuracy_rocks': 0.19915371534780718,
 'eval_accuracy_pool': 0.9833262441514249,
 'eval_accuracy_vegetation': 0.8537524908533004,
 'eval_accuracy_roof': 0.9326926157296171,
 'eval_accuracy_wall': 0.645694314386203,
 'eval_accuracy_window': 0.04050680168933896,
 'eval_accuracy_door': 0.0,
 'eval_accuracy_fence': 0.2970566142168103,
 'eval_accuracy_fence-pole': 0.0,
 'eval_accuracy_person': 0.6784083392440169,
 'eval_accuracy_dog': nan,
 'eval_accuracy_car': 0.9221640425416266,
 'eval_accuracy_bicycle': 0.25326212630072126,
 'eval_accuracy_tree': 0.7187087792868467,
 'ev

In [14]:
MODEL_TYPE = "segformer_b2"
IMAGE_DIR = r"D:\Work\study_full_data\image_segmentation\Semantic_Segmentation_Drone_Dataset\semantic_drone_dataset\semantic_drone_dataset\original_images"
LABEL_DIR = r"D:\Work\study_full_data\image_segmentation\Semantic_Segmentation_Drone_Dataset\semantic_drone_dataset\semantic_drone_dataset\label_images_semantic"
OUTPUT_DIR = r"D:\Work\study_full_data\image_segmentation\output\segformer_b2"
EPOCHS = 50
BATCH_SIZE = 4
LEARNING_RATE = 3e-5

In [15]:
train()


  Model: segformer_b2
  Epochs: 50
  Batch size: 4
  Learning rate: 3e-05
  Image size: (512, 512)

Creating dataset...
Loaded 400 image-label pairs


Saving the dataset (0/1 shards):   0%|          | 0/400 [00:00<?, ? examples/s]

Dataset cached to D:\Work\study_full_data\image_segmentation\output\segformer_b2\dataset_cache.hf
Train: 320, Val: 40, Test: 40
Split info saved to D:\Work\study_full_data\image_segmentation\output\segformer_b2\split_info.json


Some weights of SegformerForSemanticSegmentation were not initialized from the model checkpoint at nvidia/mit-b2 and are newly initialized: ['decode_head.batch_norm.bias', 'decode_head.batch_norm.num_batches_tracked', 'decode_head.batch_norm.running_mean', 'decode_head.batch_norm.running_var', 'decode_head.batch_norm.weight', 'decode_head.classifier.bias', 'decode_head.classifier.weight', 'decode_head.linear_c.0.proj.bias', 'decode_head.linear_c.0.proj.weight', 'decode_head.linear_c.1.proj.bias', 'decode_head.linear_c.1.proj.weight', 'decode_head.linear_c.2.proj.bias', 'decode_head.linear_c.2.proj.weight', 'decode_head.linear_c.3.proj.bias', 'decode_head.linear_c.3.proj.weight', 'decode_head.linear_fuse.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model parameters: 27,365,080


...


  Final Evaluation on Test Set (unseen during training)


C:\Users\fbwlg\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-metric--mean_iou\9e450724f21f05592bfb0255fe2fa576df8171fa060d11121d8aecfff0db80d0\mean_iou.py:259: RuntimeWarning: invalid value encountered in divide
  iou = total_area_intersect / total_area_union
C:\Users\fbwlg\.cache\huggingface\modules\evaluate_modules\metrics\evaluate-metric--mean_iou\9e450724f21f05592bfb0255fe2fa576df8171fa060d11121d8aecfff0db80d0\mean_iou.py:260: RuntimeWarning: invalid value encountered in divide
  acc = total_area_intersect / total_area_label



  Mean IoU:      0.6279
  Mean Accuracy: 0.7308
  Overall Acc:   0.9048

  Category             IoU   Accuracy
  -----------------------------------
  unlabeled            nan        nan
  paved-area        0.9157     0.9515
  dirt              0.5818     0.6835
  grass             0.9264     0.9628
  gravel            0.7298     0.8741
  water             0.9020     0.9545
  rocks             0.4366     0.6853
  pool              0.9466     0.9846
  vegetation        0.7581     0.8864
  roof              0.9451     0.9766
  wall              0.5794     0.7033
  window            0.4185     0.6554
  door              0.0000     0.0000
  fence             0.4610     0.6260
  fence-pole        0.0000     0.0000
  person            0.6631     0.8099
  dog                  nan        nan
  car               0.8984     0.9536
  bicycle           0.4243     0.4960
  tree              0.6691     0.7693
  bald-tree         0.5644     0.8226
  ar-marker         0.8106     0.8651
  obstacle    

D:\ProgramData\anaconda3\envs\transformer_learn\Lib\site-packages\transformers\utils\deprecation.py:172: UserWarning: The following named arguments are not valid for `SegformerImageProcessor.__init__` and were ignored: 'feature_extractor_type'
  return func(*args, **kwargs)


{'eval_loss': 0.30598777532577515,
 'eval_mean_iou': 0.6278615485828836,
 'eval_mean_accuracy': 0.7307702620076638,
 'eval_overall_accuracy': 0.9047899385913145,
 'eval_accuracy_unlabeled': nan,
 'eval_accuracy_paved-area': 0.9514753628370202,
 'eval_accuracy_dirt': 0.683452404129321,
 'eval_accuracy_grass': 0.9628104043918381,
 'eval_accuracy_gravel': 0.8741104489471593,
 'eval_accuracy_water': 0.9544736674520792,
 'eval_accuracy_rocks': 0.685253292205481,
 'eval_accuracy_pool': 0.9846022968949383,
 'eval_accuracy_vegetation': 0.8864200503944406,
 'eval_accuracy_roof': 0.9765897805542207,
 'eval_accuracy_wall': 0.7033099105608931,
 'eval_accuracy_window': 0.6554355181183937,
 'eval_accuracy_door': 0.0,
 'eval_accuracy_fence': 0.6260377955246303,
 'eval_accuracy_fence-pole': 0.0,
 'eval_accuracy_person': 0.809856629396154,
 'eval_accuracy_dog': nan,
 'eval_accuracy_car': 0.9535629518236008,
 'eval_accuracy_bicycle': 0.4960083686615647,
 'eval_accuracy_tree': 0.769271421880975,
 'eval_a